# L13 – Attention och transformers: grundidén

**Tid:** ca 2 h · **Del C: Nätverkstyper**

## Mål
1. Förstå vad **attention** gör: varje token tittar på alla andra och väljer vilka som är relevanta.
2. Räkna attention för hand med **query, key och value**.
3. Förstå **varför transformers har ersatt RNN** för text, och varför de behöver **positionsinformation**.
4. Bygga en liten textklassificerare med `nn.TransformerEncoder` och jämföra den med en **TF-IDF-baslinje**.

**Var i arbetsflödet?** Steg 3–6 för text: baslinje, datapipeline (tokenisering) och modell.

> Kräver `dlkit` från L09. Första körningen laddar ned IMDB (cirka 80 MB). **Körtid:** cirka 3–4 minuter.

In [ ]:
import sys
sys.path.insert(0, "..")

import re
import math
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader

from dlkit.seed import set_seed
from dlkit.train import fit, evaluate
from dlkit.plots import plot_history

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

## 1. Idén: vilka ord är relevanta för vilka?

Ta meningen *"Filmen var inte särskilt bra"*. För att förstå **"bra"** måste modellen koppla det till **"inte"**, som står två ord bort. En RNN (L12) måste bära med sig "inte" i sitt minne genom varje steg. **Attention** låter i stället "bra" titta direkt på alla ord i meningen och själv välja vilka som spelar roll.

### Query, key och value: en sökning
Varje token (ord) gör om sin vektor till tre nya vektorer med tre inlärda `Linear`-lager:
- **Query** $q$: "vad letar jag efter?"
- **Key** $k$: "vad innehåller jag?"
- **Value** $v$: "vad lämnar jag vidare om någon väljer mig?"

Likheten mellan en query och alla keys avgör **hur mycket** varje token ska bidra. Utdata blir ett viktat medelvärde av values:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d}}\right) V$$

Det är som att söka i en databas. Query är söktermen, keys är indexet och values är innehållet. Skillnaden är att sökningen är "mjuk": du får en blandning av alla träffar, viktad efter hur väl de matchar.

## 2. Räkna för hand

Vi tar tre tokens med vektorer i 2 dimensioner. För att hålla det enkelt låter vi $Q = K = V = X$, utan inlärda vikter.

In [ ]:
tokens = ["inte", "bra", "film"]
X = torch.tensor([[1.0, 0.0],     # "inte"
                  [0.8, 0.6],     # "bra"
                  [0.0, 1.0]])    # "film"
Q, K, V = X, X, X
d = Q.shape[-1]

scores = Q @ K.T / math.sqrt(d)          # (3, 3): hur väl varje query matchar varje key
weights = torch.softmax(scores, dim=-1)  # varje rad summerar till 1
output = weights @ V                     # viktat medel av values

print("scores:\n", scores.round(decimals=3))
print("attention-vikter (rad = vem som tittar, kolumn = på vem):\n", weights.round(decimals=3))
print("utdata:\n", output.round(decimals=3))

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3.5))
ax.imshow(weights, cmap="Blues")
ax.set_xticks(range(3), tokens); ax.set_yticks(range(3), tokens)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{weights[i, j]:.2f}", ha="center", va="center")
ax.set_xlabel("key (tittar på)"); ax.set_ylabel("query (den som tittar)"); plt.show()

Raden för "bra" visar att den tittar mest på sig själv, men också på "inte", eftersom vektorerna liknar varandra. Utdata för "bra" är alltså en **blandning** som innehåller information om "inte". Det är kontext.

**Varför dela med $\sqrt{d}$?** Skalärprodukter växer med dimensionen. Utan skalning blir talen i softmax stora, softmax blir nästan "allt på en", och gradienterna blir små (mättnad, som för sigmoid i L05).

PyTorch har en färdig, snabb funktion:

In [ ]:
print(F.scaled_dot_product_attention(Q[None], K[None], V[None])[0].round(decimals=3))

### Multi-head attention
En "head" kan bara lära sig en typ av relation. **Multi-head attention** kör flera attention-beräkningar parallellt, var och en med egna query-, key- och value-lager, och slår ihop resultaten. En head kan till exempel lära sig negation och en annan vilket substantiv ett adjektiv hör till. `nn.MultiheadAttention` och `nn.TransformerEncoderLayer` sköter det.

## 3. Varför har transformers ersatt RNN?

| | RNN och LSTM (L12) | Transformer |
|---|---|---|
| Avstånd mellan två ord | informationen går genom alla steg emellan | **direkt**, ett steg |
| Beräkning | sekventiell: steg *t* väntar på *t−1* | **parallell** över alla positioner, vilket passar GPU |
| Långa sekvenser | glömmer trots grindar | bra, men kostnaden växer kvadratiskt med längden |
| Ordningsinformation | inbyggd | **måste läggas till** (positionsembedding) |

### Attention bryr sig inte om ordning
Attention är ett viktat medelvärde, och ett medelvärde bryr sig inte om ordningen. Om vi kastar om tokens kastas utdata om på samma sätt, men *värdet* för varje token blir detsamma:

In [ ]:
perm = torch.tensor([2, 0, 1])
out_original = F.scaled_dot_product_attention(X[None], X[None], X[None])[0]
out_permuted = F.scaled_dot_product_attention(X[perm][None], X[perm][None], X[perm][None])[0]
print("samma utdata per token oavsett ordning:", torch.allclose(out_original[perm], out_permuted))

"Hunden bet mannen" och "mannen bet hunden" skulle alltså se likadana ut. Därför adderar man en **positionsembedding**, en inlärd vektor per position (0, 1, 2, …), till varje tokens vektor. Då vet modellen var i meningen ordet står.

## 4. Data: filmrecensioner från IMDB

50 000 recensioner, märkta som positiva eller negativa, med 25 000 för träning och 25 000 för test. Vi delar träningsdelen i 20 000 train och 5 000 val.

In [ ]:
from datasets import load_dataset

imdb = load_dataset("stanfordnlp/imdb", cache_dir="../data/hf")
train_all = imdb["train"].shuffle(seed=42)
train_texts, train_labels = train_all["text"][:20_000], train_all["label"][:20_000]
val_texts, val_labels = train_all["text"][20_000:], train_all["label"][20_000:]
test_texts, test_labels = imdb["test"]["text"], imdb["test"]["label"]
print(f"train {len(train_texts)}, val {len(val_texts)}, test {len(test_texts)}")
print("andel positiva i train:", np.mean(train_labels))
print("\nexempel:", train_texts[0][:300], "...")

### Baslinje: TF-IDF och logistisk regression
Det här känner du igen från sklearn. Varje recension blir en vektor med ordfrekvenser (*bag of words*), och ordningen ignoreras helt. Det är en mycket stark baslinje för sentimentanalys.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tfidf = TfidfVectorizer(max_features=20_000)
clf = LogisticRegression(max_iter=1000).fit(tfidf.fit_transform(train_texts), train_labels)
tfidf_val_acc = clf.score(tfidf.transform(val_texts), val_labels)
print(f"TF-IDF + LogisticRegression, val accuracy: {tfidf_val_acc:.3f}")

## 5. Från text till tensorer: tokenisering

Ett neuralt nät behöver tal. Stegen är:
1. **Tokenisera:** dela texten i ord (tokens). Vi använder ett enkelt regex. Riktiga modeller använder *subword*-tokenisering (se "Senare fördjupning").
2. **Vokabulär:** de 20 000 vanligaste orden **i train** får ett id. Två specialtoken finns: `<pad>` (id 0) och `<unk>` (id 1, okända ord).
3. **Kapa och fyll ut:** alla sekvenser måste ha samma längd i en batch. Vi kapar efter 256 tokens och fyller ut korta recensioner med `<pad>`.

In [ ]:
def tokenize(text):
    return re.findall(r"[a-z0-9']+", text.lower().replace("<br />", " "))

counts = Counter(token for text in train_texts for token in tokenize(text))   # bara train!
itos = ["<pad>", "<unk>"] + [w for w, _ in counts.most_common(20_000)]          # id → ord
stoi = {w: i for i, w in enumerate(itos)}                                       # ord → id
MAX_LEN = 256

def encode(text):
    ids = [stoi.get(tok, 1) for tok in tokenize(text)][:MAX_LEN]
    return ids + [0] * (MAX_LEN - len(ids))

print("tokens:", tokenize(train_texts[0])[:10])
print("ids:   ", encode(train_texts[0])[:10])
lengths = [len(tokenize(t)) for t in train_texts[:2000]]
print(f"medianlängd {np.median(lengths):.0f} tokens, {np.mean(np.array(lengths) > MAX_LEN):.0%} kapas")

In [ ]:
def to_loader(texts, labels, batch_size, shuffle):
    X = torch.tensor([encode(t) for t in texts])
    y = torch.tensor(labels)
    return DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=shuffle)

train_loader = to_loader(train_texts, train_labels, 64, shuffle=True)
val_loader = to_loader(val_texts, val_labels, 256, shuffle=False)
print("batch:", next(iter(train_loader))[0].shape)   # (64, 256): 64 recensioner, 256 token-id

## 6. En liten transformer för textklassificering

| Del | Vad | Varför |
|---|---|---|
| `nn.Embedding(vocab, d_model)` | gör om varje token-id till en inlärd vektor | ett id (till exempel 523) har ingen betydelse i sig. Vektorn lär sig betydelsen. |
| `nn.Embedding(MAX_LEN, d_model)` | positionsembedding, adderas | attention vet annars inget om ordning (avsnitt 3) |
| `nn.TransformerEncoderLayer` | multi-head attention, sedan ett litet MLP, med **LayerNorm** och **residualkopplingar** | standardblocket. LayerNorm (L07) fungerar för varierande längd. Residualkopplingarna gör djupa staplar träningsbara. |
| `src_key_padding_mask` | säger vilka positioner som är `<pad>` | så att ingen token "tittar på" utfyllnaden |
| medelvärde över tokens och sedan `Linear` | en vektor per recension, sedan 2 logits | klassificeringshuvud, som i L10 |

In [ ]:
class TextTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=64, nhead=4, num_layers=2, max_len=MAX_LEN,
                 num_classes=2, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, d_model)
        layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=4 * d_model, dropout=dropout,
                                           activation="gelu", batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers, enable_nested_tensor=False)
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, x):                                   # x: (N, L) token-id
        pad = x == 0                                        # True där det är utfyllnad
        positions = torch.arange(x.size(1), device=x.device)
        h = self.tok_emb(x) + self.pos_emb(positions)       # (N, L, d_model)
        h = self.encoder(h, src_key_padding_mask=pad)       # attention mellan alla riktiga tokens
        keep = (~pad).unsqueeze(-1).float()                 # (N, L, 1)
        h = (h * keep).sum(dim=1) / keep.sum(dim=1).clamp(min=1)   # medel över riktiga tokens
        return self.head(h)                                 # (N, 2) logits

set_seed(42)
model = TextTransformer(len(itos)).to(device)
n_emb = model.tok_emb.weight.numel()
n_total = sum(p.numel() for p in model.parameters())
print(f"{n_total:,} parametrar, varav {n_emb:,} ({n_emb / n_total:.0%}) i token-embeddingen")

**Valen:**
- `d_model=64` och `nhead=4` ger 16 dimensioner per head. Det är litet med flit, så att det går snabbt.
- `num_layers=2` räcker för en klassificeringsuppgift.
- **GELU** är standard i transformers (L05).
- `norm_first=True` lägger LayerNorm *före* attention. Det ger stabilare träning och är vanligt i moderna modeller.

In [ ]:
h_tr = fit(model, train_loader, val_loader, nn.CrossEntropyLoss(),
           torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01), device,
           epochs=8, patience=2)
plot_history(h_tr, "Liten transformer på IMDB")

In [ ]:
print(f"TF-IDF + LogisticRegression: {tfidf_val_acc:.3f}")
print(f"Liten transformer:           {max(h_tr['val_acc']):.3f}")

## 7. Ärlig jämförelse: varför vinner baslinjen?

Den lilla transformern hamnar runt **85–86 %**, *under* TF-IDF-baslinjen (cirka 89 %), och den börjar överanpassa efter några epoker. Det är inget misslyckande, utan en viktig lärdom:
- **Lite data för modellen:** 20 000 recensioner räcker inte för att lära sig ordens betydelse *och* hur de samverkar från grunden. Embeddingen ensam har 1,3 miljoner parametrar.
- **Uppgiften passar bag of words:** sentiment avgörs i hög grad av *vilka* ord som finns ("awful", "brilliant"). Där är TF-IDF stark.
- **Transformers vinner med förträning:** modeller som BERT förtränas på miljarder ord *innan* de finjusteras på IMDB. De når cirka 94–95 %. Det är samma princip som transfer learning för bilder (L11).

**Slutsats för arbetsflödet:** börja alltid med en enkel baslinje. För text med begränsad data finjusterar du en *förtränad* transformer, till exempel via biblioteket Hugging Face `transformers`, i stället för att träna en från grunden.

## 8. Test – en gång

In [ ]:
test_loader = to_loader(test_texts, test_labels, 256, shuffle=False)
print(f"TEST – transformer: {evaluate(model, test_loader, nn.CrossEntropyLoss(), device)[1]:.3f}   "
      f"TF-IDF: {clf.score(tfidf.transform(test_texts), test_labels):.3f}")

## 9. Övningar

### Övning 1 – attention för hand
Skriv `my_attention(Q, K, V)` utan `F.scaled_dot_product_attention`. Den ska fungera för shapes `(N, L, d)`.

*Ledtråd:* `K.transpose(-2, -1)` byter plats på de två sista dimensionerna.

In [ ]:
def my_attention(Q, K, V):
    # TODO
    ...

In [ ]:
set_seed(0)
Qt, Kt, Vt = torch.randn(2, 5, 8), torch.randn(2, 5, 8), torch.randn(2, 5, 8)
assert torch.allclose(my_attention(Qt, Kt, Vt), F.scaled_dot_product_attention(Qt, Kt, Vt), atol=1e-6)
print("Rätt!")

### Övning 2 – utan positionsembedding
Gör en variant av `TextTransformer` **utan** `pos_emb` och visa att modellen ger *exakt samma* utdata när orden i en recension kastas om. Varför spelar det mindre roll för just sentiment än för till exempel översättning?

In [ ]:
# TODO: skapa NoPosTransformer, sätt den i eval-läge och jämför utdata för x och x med omkastade (icke-pad) tokens
same_output = None

In [ ]:
assert same_output is True
print("Rätt!")

*Ditt svar:*

### Övning 3 – storleken på modellen
Testa `d_model=128` och `num_layers=4`. Blir den bättre eller sämre på val, och överanpassar den tidigare? Koppla till diagnosen i L06.

In [ ]:
# TODO

## När använder jag detta?
- **Attention och transformers:** standard för text, och allt vanligare för bilder (Vision Transformer), ljud och tidsserier, särskilt med **mycket data eller förträning**.
- **Förtränade transformers** (BERT, GPT och liknande): förstahandsvalet för textuppgifter. Finjustera dem i stället för att träna från grunden.
- **TF-IDF och logistisk regression:** den första baslinjen för textklassificering. Den är snabb och förvånansvärt stark.
- **LSTM och GRU:** för korta till medellånga sekvenser med lite data, eller när beräkningskraften är begränsad.

## Sammanfattning
- Attention: varje token beräknar ett viktat medel över alla tokens: softmax(QKᵀ/√d)·V.
- Transformers är parallella och kopplar ihop ord direkt, men behöver positionsembedding.
- Ett transformerblock består av multi-head attention, ett MLP, LayerNorm och residualkopplingar.
- Från grunden på 20 000 recensioner slår transformern *inte* TF-IDF. Styrkan kommer med förträning.

**Del C är klar.** Nästa lektion: samma arbetsflöde i TensorFlow/Keras.